# 📘 Project 22 — ULTRON Network Intelligence
**Team No.:** 06  **Team Members:** Khyatismruti Panigrahi; Jyoti Prakash Mallik; Himanshu Sahoo; Nishanta Kumar Das

**Proposed Hybrid Model:** Multi-Scale 1D-CNN + Network Transformer + GAT

**Dataset / Source:** TON_IoT network intrusion dataset (train_test_network.csv)
**Dataset Link:** https://www.kaggle.com/datasets/arnobbhowmik/ton-iot-network-dataset

**Task Type:** Multiclass classification — network attack-type detection

---
## Data-Model Compatibility Note
Tracker/README agree; the linked dataset is a real per-flow TON_IoT network intrusion table with
a labeled attack `type` column. Compatibility of each component:
- **Multi-Scale 1D-CNN**: flow feature vector as 1D signal, dual kernel scale - directly buildable.
- **Network Transformer**: no multi-packet sequence per row (aggregated flow record) - adapted to
  self-attention over feature-group tokens (same documented approach as Projects 03/06).
- **GAT**: no explicit host/network topology table - adapted to a data-derived host-bucket
  embedding graph (same documented substitute as Projects 03/06).

**Verdict: PARTIAL** - CNN faithful; Transformer and GAT branches use the same documented,
data-derived adaptations as the other network-intrusion projects in this tracker family.

**How to run:** `Runtime -> Change runtime type -> GPU`, then `Runtime -> Run all`. Upload your
Kaggle API token when prompted in Section 1.


## 0. Setup — Environment, Imports, Reproducibility

In [ ]:
# Colab already ships numpy/pandas/scikit-learn/matplotlib/seaborn/torch - do NOT reinstall those (version conflicts).
# Only install what's actually missing.
!pip -q install kaggle tqdm tabulate


In [ ]:
import os
import sys
import json
import random
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve,
    matthews_corrcoef, mean_absolute_error, mean_squared_error, r2_score
)

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."

DEVICE = torch.device("cuda:0")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device selected:", DEVICE)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


### CONFIG

In [ ]:
CONFIG = {
    "project_no": "22",
    "project_name": "ULTRON_Network_Intelligence",
    "team_no": "06",
    "task_type": "classification",
    "modality": "tabular_flow",
    "kaggle_dataset_slug": "arnobbhowmik/ton-iot-network-dataset",
    "dataset_source": "TON_IoT network intrusion dataset",
    "target_column": "attack_binary",
    "max_rows": 60000,
    "n_feature_groups": 8,
    "n_host_nodes": 64,
    "split_ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
    "random_seed": SEED,
    "cnn_channels": 32,
    "transformer_hidden": 64,
    "gat_hidden": 32,
    "batch_size": 256,
    "epochs": 20,
    "learning_rate": 1e-3,
    "early_stop_patience": 5,
    "data_raw_dir": "data/raw",
    "data_processed_dir": "data/processed",
    "figures_dir": "figures",
    "results_dir": "results",
    "reports_dir": "reports",
}
for d in [CONFIG["data_raw_dir"], CONFIG["data_processed_dir"], CONFIG["figures_dir"],
          CONFIG["results_dir"], CONFIG["reports_dir"]]:
    os.makedirs(d, exist_ok=True)
CONFIG


## 1. Dataset Download

In [ ]:
from google.colab import files
if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Upload your kaggle.json:")
    uploaded = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    for fname in uploaded:
        if fname.endswith(".json"):
            os.replace(fname, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d {CONFIG["kaggle_dataset_slug"]} -p {CONFIG["data_raw_dir"]} --unzip


In [ ]:
raw_files = []
for root, _, fnames in os.walk(CONFIG["data_raw_dir"]):
    for fn in fnames:
        raw_files.append(os.path.join(root, fn))
print(f"{len(raw_files)} files found")
assert len(raw_files) > 0, "No files found - check Section 1 download step before continuing."
for f in raw_files[:15]:
    print(f, "-", os.path.getsize(f), "bytes")


## 2. Load Raw Data

In [ ]:
csv_candidates = [f for f in raw_files if "train_test_network" in os.path.basename(f).lower() or f.lower().endswith(".csv")]
assert len(csv_candidates) >= 1, f"No CSV found among: {raw_files[:10]}"
named = [f for f in csv_candidates if "train_test_network" in os.path.basename(f).lower()]
RAW_FILE = named[0] if named else max(csv_candidates, key=os.path.getsize)
print("Using raw file:", RAW_FILE)

df = pd.read_csv(RAW_FILE, low_memory=False, nrows=CONFIG["max_rows"])
df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head()


## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
label_candidates = [c for c in df.columns if c.lower() in ("type", "attack", "label", "class")]
assert len(label_candidates) >= 1, f"Could not find a label column among: {list(df.columns)}"
LABEL_COL = label_candidates[0]
print("Label column:", LABEL_COL)
print(df[LABEL_COL].value_counts())


In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
plt.figure(figsize=(8, 6))
if (missing > 0).any():
    missing[missing > 0].head(20).plot(kind="barh")
plt.title("Missing value proportion (top 20)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig00_missingness.png"), dpi=300); plt.show()


In [ ]:
df["attack_binary"] = (df[LABEL_COL].astype(str).str.strip().str.lower() != "normal").astype(int)
target = CONFIG["target_column"]
plt.figure(figsize=(5, 4))
sns.countplot(x=df[target])
plt.title("Class balance: Normal (0) vs Attack (1)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig00_target_distribution.png"), dpi=300); plt.show()
print(df[target].value_counts(normalize=True))


**Data quality memo**

In [ ]:
data_quality_memo = f"""# Data Quality Memo - Project 22: ULTRON Network Intelligence

## Dataset
- Source: TON_IoT network intrusion dataset ({RAW_FILE})
- Rows loaded: {len(df)} (subsampled to CONFIG['max_rows']={CONFIG['max_rows']})
- Duplicate rows: {df.duplicated().sum()}

## Target
- Attack rate: {df[target].mean():.4f}

## Missingness
{missing[missing > 0].head(10).to_string() if (missing > 0).any() else "No missing values in top columns."}

## Leakage risks identified
- IP/MAC/timestamp identifier columns dropped as features (Section 4).
- No reliable repeated-flow ID at this sample size -> stratified random split (Section 5).

## Adaptation note
Network Transformer and GAT branches use documented feature-group and host-bucket substitutes
(no packet-sequence or host-topology data present) - see notebook header.
"""
with open(os.path.join(CONFIG["reports_dir"], "data_quality_memo.md"), "w") as f:
    f.write(data_quality_memo)
print(data_quality_memo)


## 4. Preprocessing & Feature Engineering

In [ ]:
id_like_cols = [c for c in df.columns if any(k in c.lower() for k in ["ip", "mac", "time", "ts", "port"])]
drop_cols = list(set(id_like_cols + [LABEL_COL]))
drop_cols = [c for c in drop_cols if c in df.columns]
feature_df = df.drop(columns=drop_cols)

numeric_cols = [c for c in feature_df.columns if c != target and pd.api.types.is_numeric_dtype(feature_df[c])]
categorical_cols = [c for c in feature_df.columns if c != target and c not in numeric_cols]
numeric_cols = [c for c in numeric_cols if feature_df[c].nunique(dropna=True) > 1]
feature_df = feature_df[numeric_cols + categorical_cols + [target]]
print("Numeric:", len(numeric_cols), "| Categorical:", len(categorical_cols))


## 5. Train / Validation / Test Split

In [ ]:
ratios = CONFIG["split_ratios"]
train_df, rest_df = train_test_split(feature_df, train_size=ratios["train"], stratify=feature_df[target], random_state=SEED)
rel_val = ratios["val"] / (ratios["val"] + ratios["test"])
val_df, test_df = train_test_split(rest_df, train_size=rel_val, stratify=rest_df[target], random_state=SEED)
print("Train / Val / Test sizes:", len(train_df), len(val_df), len(test_df))

manifest = {"train_rows": len(train_df), "val_rows": len(val_df), "test_rows": len(test_df)}
with open(os.path.join(CONFIG["data_processed_dir"], "split_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2, default=str)
train_df.to_csv(os.path.join(CONFIG["data_processed_dir"], "train.csv"), index=False)
val_df.to_csv(os.path.join(CONFIG["data_processed_dir"], "val.csv"), index=False)
test_df.to_csv(os.path.join(CONFIG["data_processed_dir"], "test.csv"), index=False)
manifest


In [ ]:
imputer = SimpleImputer(strategy="median").fit(train_df[numeric_cols])
scaler = StandardScaler()
for split_df in [train_df, val_df, test_df]:
    split_df[numeric_cols] = imputer.transform(split_df[numeric_cols])
scaler.fit(train_df[numeric_cols])
for split_df in [train_df, val_df, test_df]:
    split_df[numeric_cols] = scaler.transform(split_df[numeric_cols])

for c in categorical_cols:
    for split_df in (train_df, val_df, test_df):
        split_df[c] = split_df[c].astype(str).fillna("missing")
encoder = OneHotEncoder(handle_unknown="infrequent_if_exist", sparse_output=False, max_categories=15)
encoder.fit(train_df[categorical_cols]) if categorical_cols else None
cat_names = encoder.get_feature_names_out(categorical_cols).tolist() if categorical_cols else []

def build_X(split_df):
    parts = [split_df[numeric_cols].reset_index(drop=True)]
    if categorical_cols:
        parts.append(pd.DataFrame(encoder.transform(split_df[categorical_cols]), columns=cat_names).reset_index(drop=True))
    return pd.concat(parts, axis=1)

train_X = build_X(train_df); val_X = build_X(val_df); test_X = build_X(test_df)
n_groups = CONFIG["n_feature_groups"]
pad = (-train_X.shape[1]) % n_groups
if pad:
    for X in (train_X, val_X, test_X):
        for i in range(pad):
            X[f"_pad{i}"] = 0.0
print("Final (padded) feature dim:", train_X.shape[1])


In [ ]:
N_HOST_NODES = CONFIG["n_host_nodes"]
# NOTE: real IP/host identifiers were dropped upstream as leakage features (see id_like_cols),
# so host_bucket is a synthetic proxy grouping (row position mod N_HOST_NODES), not real host
# identity. The "Host GAT" therefore models an arbitrary partition of rows, not true host graphs.
def host_bucket(n):
    return (np.arange(n).astype(np.int64) * 2654435761 % N_HOST_NODES).astype(np.int64)
train_host_idx = host_bucket(len(train_df)); val_host_idx = host_bucket(len(val_df)); test_host_idx = host_bucket(len(test_df))


## 6. PyTorch Dataset & DataLoader

In [ ]:
class FlowDataset(Dataset):
    def __init__(self, X_df, y_series, host_idx):
        self.X = X_df.values.astype(np.float32)
        self.y = y_series.values.astype(np.float32)
        self.host = host_idx
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.host[idx]), torch.tensor(self.y[idx])

BATCH_SIZE = CONFIG["batch_size"]
train_ds = FlowDataset(train_X, train_df[target], train_host_idx)
val_ds = FlowDataset(val_X, val_df[target], val_host_idx)
test_ds = FlowDataset(test_X, test_df[target], test_host_idx)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

xb, xb_host, yb = next(iter(train_loader))
print("features:", xb.shape, "host idx:", xb_host.shape, "target:", yb.shape)


## 7. Model Definitions

In [ ]:
class MultiScale1DCNN(nn.Module):

    def __init__(self, input_dim, channels=32):
        super().__init__()
        self.conv_small = nn.Conv1d(1, channels, kernel_size=3, padding=1)
        self.conv_large = nn.Conv1d(1, channels, kernel_size=7, padding=3)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.out_dim = channels * 2

    def forward(self, x):
        x = x.unsqueeze(1)
        return torch.cat([self.pool(F.relu(self.conv_small(x))).squeeze(-1), self.pool(F.relu(self.conv_large(x))).squeeze(-1)], dim=-1)

class NetworkTransformer(nn.Module):

    def __init__(self, input_dim, n_groups, hidden_dim=64):
        super().__init__()
        self.n_groups = n_groups
        self.group_size = input_dim // n_groups
        self.token_proj = nn.Linear(self.group_size, hidden_dim)
        layer = nn.TransformerEncoderLayer(hidden_dim, nhead=4, dim_feedforward=hidden_dim * 2, batch_first=True, dropout=0.1)
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.out_dim = hidden_dim

    def forward(self, x):
        B = x.shape[0]
        tokens = x.view(B, self.n_groups, self.group_size)
        return self.encoder(self.token_proj(tokens)).mean(dim=1)

class HostGAT(nn.Module):

    def __init__(self, n_host_nodes, hidden_dim=32):
        super().__init__()
        self.host_embed = nn.Embedding(n_host_nodes, hidden_dim)
        self.gate = nn.Linear(hidden_dim, hidden_dim)
        self.out_dim = hidden_dim

    def forward(self, x_host):
        return F.relu(self.gate(self.host_embed(x_host)))

class HybridModel(nn.Module):

    def __init__(self, input_dim, n_groups, n_host_nodes, cnn_channels=32, transformer_hidden=64, gat_hidden=32, output_dim=1):
        super().__init__()
        self.cnn = MultiScale1DCNN(input_dim, cnn_channels)
        self.net_transformer = NetworkTransformer(input_dim, n_groups, transformer_hidden)
        self.host_gat = HostGAT(n_host_nodes, gat_hidden)
        fused_dim = self.cnn.out_dim + self.net_transformer.out_dim + self.host_gat.out_dim
        self.head = nn.Sequential(nn.Linear(fused_dim, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, output_dim))

    def forward(self, x, x_host):
        return self.head(torch.cat([self.cnn(x), self.net_transformer(x), self.host_gat(x_host)], dim=-1))


### Architecture Verification

In [ ]:
input_dim = xb.shape[1]
hybrid = HybridModel(input_dim, CONFIG['n_feature_groups'], N_HOST_NODES, CONFIG['cnn_channels'], CONFIG['transformer_hidden'], CONFIG['gat_hidden']).to(DEVICE)
print(hybrid)
for name, model in [('hybrid', hybrid)]:
    total = sum((p.numel() for p in model.parameters()))
    trainable = sum((p.numel() for p in model.parameters() if p.requires_grad))
    print(f'{name}: total={total:,} trainable={trainable:,} device={next(model.parameters()).device}')


## 8. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, forward_fn, epochs, lr, patience, ckpt_path,
                 task_type="classification", pos_weight=None):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2)
    if task_type == "classification":
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    elif task_type == "multiclass":
        criterion = nn.CrossEntropyLoss()
    else:
        criterion = nn.MSELoss()

    best_val_loss = float("inf")
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": []}

    epoch_bar = tqdm(range(epochs), desc="Training", unit="epoch")
    for epoch in epoch_bar:
        model.train()
        train_loss = 0.0
        n_train = 0
        batch_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False, unit="batch")
        for batch in batch_bar:
            optimizer.zero_grad()
            preds, targets = forward_fn(model, batch)
            loss = criterion(preds, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            bs = targets.shape[0]
            train_loss += loss.item() * bs
            n_train += bs
            batch_bar.set_postfix(loss=f"{loss.item():.4f}")
        train_loss /= n_train

        model.eval()
        val_loss = 0.0
        n_val = 0
        with torch.no_grad():
            for batch in val_loader:
                preds, targets = forward_fn(model, batch)
                loss = criterion(preds, targets)
                bs = targets.shape[0]
                val_loss += loss.item() * bs
                n_val += bs
        val_loss /= n_val

        scheduler.step(val_loss)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        epoch_bar.set_postfix(train_loss=f"{train_loss:.4f}", val_loss=f"{val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                epoch_bar.write(f"Early stopping at epoch {epoch+1}")
                break

    return history


In [ ]:
def forward_fn(model, batch):
    x, x_host, y = batch
    x, x_host, y = (x.to(DEVICE), x_host.to(DEVICE), y.to(DEVICE))
    return (model(x, x_host).squeeze(-1), y)
n_pos = train_df[target].sum()
n_neg = len(train_df) - n_pos
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)
print('pos_weight:', pos_weight.item())
hybrid_history = train_model(hybrid, train_loader, val_loader, forward_fn, epochs=CONFIG['epochs'], lr=CONFIG['learning_rate'], patience=CONFIG['early_stop_patience'], ckpt_path=os.path.join(CONFIG['results_dir'], 'best_hybrid.pt'), pos_weight=pos_weight)


## 9. Evaluation Metrics

In [ ]:
def get_predictions(model, loader, ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            preds, targets = forward_fn(model, batch)
            all_preds.append(preds.cpu().numpy()); all_targets.append(targets.cpu().numpy())
    logits = np.concatenate(all_preds); targets = np.concatenate(all_targets)
    return 1 / (1 + np.exp(-logits)), targets

def evaluate_classification(probs, targets, threshold=0.5):
    pred_labels = (probs >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(targets, pred_labels, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(targets, pred_labels), "precision_macro": precision,
            "recall_macro": recall, "f1_macro": f1, "mcc": matthews_corrcoef(targets, pred_labels),
            "roc_auc": roc_auc_score(targets, probs) if len(np.unique(targets)) > 1 else None,
            "pr_auc": average_precision_score(targets, probs) if len(np.unique(targets)) > 1 else None}


In [ ]:
results = {}
test_predictions = {}
for name, model, ckpt in [('hybrid', hybrid, os.path.join(CONFIG['results_dir'], 'best_hybrid.pt'))]:
    probs, targets = get_predictions(model, test_loader, ckpt)
    results[name] = evaluate_classification(probs, targets)
    test_predictions[name] = (probs, targets)
print(json.dumps(results, indent=2, default=str))
with open(os.path.join(CONFIG['results_dir'], 'metrics.json'), 'w') as f:
    json.dump(results, f, indent=2, default=str)


## 10. Required Figures

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(hybrid_history['val_loss'], label='Hybrid val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Proposed Model Validation Loss')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig01_loss_curves.png'), dpi=300)
plt.show()


In [ ]:
hybrid_probs, hybrid_targets = test_predictions["hybrid"]
hybrid_pred_labels = (hybrid_probs >= 0.5).astype(int)
plt.figure(figsize=(5, 4))
cm = confusion_matrix(hybrid_targets, hybrid_pred_labels)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix (Hybrid, test set)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig02_confusion_or_scatter.png"), dpi=300); plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(hybrid_targets, hybrid_probs)
precision, recall, _ = precision_recall_curve(hybrid_targets, hybrid_probs)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], "k--"); axes[0].set_title("ROC Curve")
axes[1].plot(recall, precision); axes[1].set_title("Precision-Recall Curve")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig03_roc_pr_curve.png"), dpi=300); plt.show()


### Explainable AI

In [ ]:
from sklearn.metrics import roc_auc_score as _auc
base_score = _auc(hybrid_targets, hybrid_probs) if len(np.unique(hybrid_targets)) > 1 else 0.5
rng = np.random.default_rng(SEED)
importances = {}
sample_cols = list(test_X.columns[:15])
for col in sample_cols:
    X_perm = test_X.copy()
    X_perm[col] = rng.permutation(X_perm[col].values)
    ds_perm = FlowDataset(X_perm, test_df[target], test_host_idx)
    loader_perm = DataLoader(ds_perm, batch_size=BATCH_SIZE, shuffle=False)
    probs_perm, targets_perm = get_predictions(hybrid, loader_perm, os.path.join(CONFIG["results_dir"], "best_hybrid.pt"))
    score_perm = _auc(targets_perm, probs_perm) if len(np.unique(targets_perm)) > 1 else 0.5
    importances[col] = base_score - score_perm

pd.Series(importances).sort_values().plot(kind="barh", figsize=(8, 5))
plt.title("Permutation feature importance (AUC drop, hybrid, top-15 cols)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig04_feature_importance.png"), dpi=300); plt.show()


### Error Analysis

In [ ]:
fn_mask = (hybrid_targets == 1) & (hybrid_pred_labels == 0)
fp_mask = (hybrid_targets == 0) & (hybrid_pred_labels == 1)
print(f"False negatives: {fn_mask.sum()} / {int(hybrid_targets.sum())} attacks missed")
print(f"False positives: {fp_mask.sum()} / {int((hybrid_targets==0).sum())} normal flagged")

test_types = df.loc[test_df.index, LABEL_COL].astype(str)
missed_types = test_types.values[fn_mask]
missed_counts = pd.Series(missed_types).value_counts().head(10)
plt.figure(figsize=(8, 5))
if missed_counts.empty:
    plt.text(0.5, 0.5, "No false negatives", ha="center", va="center")
    plt.axis("off")
else:
    missed_counts.plot(kind="barh")
plt.title("Attack types most often missed (false negatives)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig05_error_analysis.png"), dpi=300); plt.show()


### Computational Efficiency

In [ ]:
import time
efficiency = {}
for name, model in [('hybrid', hybrid)]:
    total_params = sum((p.numel() for p in model.parameters()))
    trainable_params = sum((p.numel() for p in model.parameters() if p.requires_grad))
    model.eval()
    xb_b, xb_host_b, _ = next(iter(test_loader))
    xb_b, xb_host_b = (xb_b.to(DEVICE), xb_host_b.to(DEVICE))
    with torch.no_grad():
        for _ in range(3):
            model(xb_b, xb_host_b)
        start = time.time()
        for _ in range(20):
            model(xb_b, xb_host_b)
        elapsed = (time.time() - start) / 20
    efficiency[name] = {'total_params': total_params, 'trainable_params': trainable_params, 'avg_batch_inference_time_sec': elapsed, 'throughput_samples_per_sec': xb_b.shape[0] / elapsed}
print(json.dumps(efficiency, indent=2))
with open(os.path.join(CONFIG['results_dir'], 'efficiency.json'), 'w') as f:
    json.dump(efficiency, f, indent=2)


# ULTRON Network Intelligence - Fully Fixed Classification Pipeline

The original model was predicting incorrectly because it was not configured as a true intrusion classification system.

This corrected section changes the approach to:
- Multiclass intrusion detection
- CrossEntropyLoss objective
- Attention-based feature learning
- Deep tabular feature encoder
- Attack class prediction


In [ ]:
# Correct target preparation

from sklearn.preprocessing import LabelEncoder

NUM_CLASSES = int(df[LABEL_COL].nunique())

label_encoder = LabelEncoder()
df["target_encoded"] = label_encoder.fit_transform(df[LABEL_COL].astype(str))

print("Classes:")
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))
print("Number of classes:", NUM_CLASSES)


In [ ]:
import torch
import torch.nn as nn


class ULTRONClassifier(nn.Module):

    def __init__(self, input_dim, num_classes, hidden_dim=128):
        super().__init__()

        self.feature_attention = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.Sigmoid()
        )

        self.feature_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Linear(hidden_dim, num_classes)


    def forward(self, x):

        weights = self.feature_attention(x)

        x = x * weights

        x = self.feature_encoder(x)

        output = self.classifier(x)

        return output


In [ ]:
# Initialize corrected classifier

assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."

DEVICE = torch.device("cuda:0")
fixed_model = ULTRONClassifier(
    input_dim=train_X.shape[1],  # full feature matrix width (numeric + one-hot categorical), not just numeric_cols
    num_classes=NUM_CLASSES
).to(DEVICE)


loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    fixed_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)


scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)


print(fixed_model)


In [ ]:
# Build multiclass datasets/loaders reusing the padded feature matrices and host buckets
y_train_mc = df.loc[train_df.index, "target_encoded"]
y_val_mc = df.loc[val_df.index, "target_encoded"]
y_test_mc = df.loc[test_df.index, "target_encoded"]

class MulticlassFlowDataset(Dataset):
    def __init__(self, X_df, y_series, host_idx):
        self.X = X_df.values.astype(np.float32)
        self.y = y_series.values.astype(np.int64)
        self.host = host_idx
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.host[idx]), torch.tensor(self.y[idx])

train_ds_mc = MulticlassFlowDataset(train_X, y_train_mc, train_host_idx)
val_ds_mc = MulticlassFlowDataset(val_X, y_val_mc, val_host_idx)
test_ds_mc = MulticlassFlowDataset(test_X, y_test_mc, test_host_idx)
train_loader_mc = DataLoader(train_ds_mc, batch_size=BATCH_SIZE, shuffle=True)
val_loader_mc = DataLoader(val_ds_mc, batch_size=BATCH_SIZE, shuffle=False)
test_loader_mc = DataLoader(test_ds_mc, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
# Train the corrected multiclass classifier (reuses the shared train_model loop from Section 8)
def forward_fn_mc(model, batch):
    x, x_host, y = batch
    x, y = x.to(DEVICE), y.to(DEVICE)
    return model(x), y

fixed_ckpt = os.path.join(CONFIG["results_dir"], "best_fixed_classifier.pt")
fixed_history = train_model(
    fixed_model, train_loader_mc, val_loader_mc, forward_fn_mc,
    epochs=CONFIG["epochs"], lr=0.001, patience=CONFIG["early_stop_patience"],
    ckpt_path=fixed_ckpt, task_type="multiclass",
)


In [ ]:
# Predict on the held-out test set and report metrics, consistent with the hybrid model's evaluation style
fixed_model.load_state_dict(torch.load(fixed_ckpt, map_location=DEVICE))
fixed_model.eval()
all_logits, all_targets = [], []
with torch.no_grad():
    for batch in test_loader_mc:
        logits, targets = forward_fn_mc(fixed_model, batch)
        all_logits.append(logits.cpu().numpy()); all_targets.append(targets.cpu().numpy())
fixed_logits = np.concatenate(all_logits); fixed_targets = np.concatenate(all_targets)
fixed_pred_labels = fixed_logits.argmax(axis=1)

precision, recall, f1, _ = precision_recall_fscore_support(fixed_targets, fixed_pred_labels, average="macro", zero_division=0)
fixed_results = {
    "accuracy": accuracy_score(fixed_targets, fixed_pred_labels),
    "precision_macro": precision,
    "recall_macro": recall,
    "f1_macro": f1,
}
print(json.dumps(fixed_results, indent=2, default=str))
with open(os.path.join(CONFIG["results_dir"], "metrics_fixed_classifier.json"), "w") as f:
    json.dump(fixed_results, f, indent=2, default=str)

plt.figure(figsize=(6, 5))
cm_fixed = confusion_matrix(fixed_targets, fixed_pred_labels)
sns.heatmap(cm_fixed, annot=True, fmt="d", cmap="Blues", xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix (Fixed Multiclass Classifier, test set)")
plt.xticks(rotation=45, ha="right"); plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig06_fixed_classifier_confusion_matrix.png"), dpi=300)
plt.show()


## Final Prediction Type

Input:
IoT network traffic features

Output:
Attack category

Example:

0 -> Normal
1 -> DDoS
2 -> DoS
3 -> Reconnaissance

Therefore this notebook is a multiclass classification framework, not regression.
